In [1]:
import pandas as pd
import numpy as np
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer
from statsmodels.tsa.seasonal import STL
from sklearn.preprocessing import StandardScaler

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
city_hour_data = '/content/drive/My Drive/impact_metrics/station_hour.csv'
df = pd.read_csv(city_hour_data, parse_dates=["Datetime"], index_col="Datetime")

/tmp/ipython-input-2254469870.py:2: DtypeWarning: Columns (15) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(city_hour_data, parse_dates=["Datetime"], index_col="Datetime")


In [4]:
df

,StationId,PM2.5,PM10,NO,NO2,NOx,NH3,CO,SO2,O3,Benzene,Toluene,Xylene,AQI,AQI_Bucket
Datetime,,,,,,,,,,,,,,,
2017-11-24 17:00:00,AP001,60.50,98.00,2.35,30.80,18.25,8.50,0.10,11.85,126.40,0.10,6.10,0.10,NaN,NaN
2017-11-24 18:00:00,AP001,65.50,111.25,2.70,24.20,15.07,9.77,0.10,13.17,117.12,0.10,6.25,0.15,NaN,NaN
2017-11-24 19:00:00,AP001,80.00,132.00,2.10,25.18,15.15,12.02,0.10,12.08,98.98,0.20,5.98,0.18,NaN,NaN
2017-11-24 20:00:00,AP001,81.50,133.25,1.95,16.25,10.23,11.58,0.10,10.47,112.20,0.20,6.72,0.10,NaN,NaN
2017-11-24 21:00:00,AP001,75.25,116.00,1.43,17.48,10.43,12.03,0.10,9.12,106.35,0.20,5.75,0.08,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2020-06-30 20:00:00,WB013,15.55,47.80,7.27,35.08,42.38,31.25,0.80,9.40,17.24,2.56,11.57,NaN,59.0,Satisfactory
2020-06-30 21:00:00,WB013,15.23,42.30,6.10,26.78,32.85,30.66,0.56,4.91,17.46,3.49,12.29,NaN,59.0,Satisfactory
2020-06-30 22:00:00,WB013,11.40,40.95,6.58,19.53,26.12,30.73,0.61,3.81,17.24,1.83,8.88,NaN,59.0,Satisfactory


In [5]:
POLLUTANTS = ['PM2.5', 'PM10', 'O3', 'NO2', 'SO2', 'CO']
df_imputed = df.copy()

# LINEAR INTERPOLATION (Gaps <= 4 hours) ---
for p in POLLUTANTS:
    df_imputed[f'{p}_Clean'] = df_imputed[p].transform(
        lambda x: x.interpolate(method='linear', limit=4, axis=0)
    )

In [6]:
df_imputed

,StationId,PM2.5,PM10,NO,NO2,NOx,NH3,CO,SO2,O3,...,Toluene,Xylene,AQI,AQI_Bucket,PM2.5_Clean,PM10_Clean,O3_Clean,NO2_Clean,SO2_Clean,CO_Clean
Datetime,,,,,,,,,,,,,,,,,,,,,
2017-11-24 17:00:00,AP001,60.50,98.00,2.35,30.80,18.25,8.50,0.10,11.85,126.40,...,6.10,0.10,NaN,NaN,60.50,98.00,126.40,30.80,11.85,0.10
2017-11-24 18:00:00,AP001,65.50,111.25,2.70,24.20,15.07,9.77,0.10,13.17,117.12,...,6.25,0.15,NaN,NaN,65.50,111.25,117.12,24.20,13.17,0.10
2017-11-24 19:00:00,AP001,80.00,132.00,2.10,25.18,15.15,12.02,0.10,12.08,98.98,...,5.98,0.18,NaN,NaN,80.00,132.00,98.98,25.18,12.08,0.10
2017-11-24 20:00:00,AP001,81.50,133.25,1.95,16.25,10.23,11.58,0.10,10.47,112.20,...,6.72,0.10,NaN,NaN,81.50,133.25,112.20,16.25,10.47,0.10
2017-11-24 21:00:00,AP001,75.25,116.00,1.43,17.48,10.43,12.03,0.10,9.12,106.35,...,5.75,0.08,NaN,NaN,75.25,116.00,106.35,17.48,9.12,0.10
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2020-06-30 20:00:00,WB013,15.55,47.80,7.27,35.08,42.38,31.25,0.80,9.40,17.24,...,11.57,NaN,59.0,Satisfactory,15.55,47.80,17.24,35.08,9.40,0.80
2020-06-30 21:00:00,WB013,15.23,42.30,6.10,26.78,32.85,30.66,0.56,4.91,17.46,...,12.29,NaN,59.0,Satisfactory,15.23,42.30,17.46,26.78,4.91,0.56
2020-06-30 22:00:00,WB013,11.40,40.95,6.58,19.53,26.12,30.73,0.61,3.81,17.24,...,8.88,NaN,59.0,Satisfactory,11.40,40.95,17.24,19.53,3.81,0.61


In [7]:
# PREVIOUS DAY'S MEAN (Gaps 5-12 hours) ---
df_imputed['Hour'] = df_imputed.index.hour
df_imputed['Date'] = df_imputed.index.date

for p in POLLUTANTS:
    clean_col = f'{p}_Clean'
    hourly_means = df_imputed.groupby(['Hour'])[clean_col].mean()
    def fill_medium_gap(row):
        if pd.isna(row[clean_col]):
            hour = row['Hour']
            return hourly_means.get((hour))
        return row[clean_col]
    df_imputed[clean_col] = df_imputed.apply(fill_medium_gap, axis=1)

In [8]:
df_imputed

,StationId,PM2.5,PM10,NO,NO2,NOx,NH3,CO,SO2,O3,...,AQI,AQI_Bucket,PM2.5_Clean,PM10_Clean,O3_Clean,NO2_Clean,SO2_Clean,CO_Clean,Hour,Date
Datetime,,,,,,,,,,,,,,,,,,,,,
2017-11-24 17:00:00,AP001,60.50,98.00,2.35,30.80,18.25,8.50,0.10,11.85,126.40,...,NaN,NaN,60.50,98.00,126.40,30.80,11.85,0.10,17,2017-11-24
2017-11-24 18:00:00,AP001,65.50,111.25,2.70,24.20,15.07,9.77,0.10,13.17,117.12,...,NaN,NaN,65.50,111.25,117.12,24.20,13.17,0.10,18,2017-11-24
2017-11-24 19:00:00,AP001,80.00,132.00,2.10,25.18,15.15,12.02,0.10,12.08,98.98,...,NaN,NaN,80.00,132.00,98.98,25.18,12.08,0.10,19,2017-11-24
2017-11-24 20:00:00,AP001,81.50,133.25,1.95,16.25,10.23,11.58,0.10,10.47,112.20,...,NaN,NaN,81.50,133.25,112.20,16.25,10.47,0.10,20,2017-11-24
2017-11-24 21:00:00,AP001,75.25,116.00,1.43,17.48,10.43,12.03,0.10,9.12,106.35,...,NaN,NaN,75.25,116.00,106.35,17.48,9.12,0.10,21,2017-11-24
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2020-06-30 20:00:00,WB013,15.55,47.80,7.27,35.08,42.38,31.25,0.80,9.40,17.24,...,59.0,Satisfactory,15.55,47.80,17.24,35.08,9.40,0.80,20,2020-06-30
2020-06-30 21:00:00,WB013,15.23,42.30,6.10,26.78,32.85,30.66,0.56,4.91,17.46,...,59.0,Satisfactory,15.23,42.30,17.46,26.78,4.91,0.56,21,2020-06-30
2020-06-30 22:00:00,WB013,11.40,40.95,6.58,19.53,26.12,30.73,0.61,3.81,17.24,...,59.0,Satisfactory,11.40,40.95,17.24,19.53,3.81,0.61,22,2020-06-30


In [9]:
# LONG GAPS (Exclusion/Flagging) ---

for p in POLLUTANTS:
    clean_col = f'{p}_Clean'
    valid_count = df_imputed.groupby(['Date'])[clean_col].count()

    # Flag days with insufficient valid data (e.g., less than 75% of a 24-hr day, or 18 hours) -- used later
    df_imputed['Hourly_Count_Per_Day'] = df_imputed.apply(
        lambda row: valid_count.get((row['Date'])), axis=1
    )
    df_imputed['Data_Valid_Flag'] = (df_imputed['Hourly_Count_Per_Day'] >= 18).astype(int)
df_hourly_cleaned = df_imputed.drop(columns=['Hour', 'Date', 'Hourly_Count_Per_Day'])

In [10]:
df_hourly_cleaned = df_hourly_cleaned.drop(columns=['PM2.5', 'PM10', 'NO', 'NO2', 'NOx', 'NH3', 'CO', 'SO2', 'O3', 'Benzene', 'Toluene', 'Xylene'])

In [11]:
df_hourly_cleaned

,StationId,AQI,AQI_Bucket,PM2.5_Clean,PM10_Clean,O3_Clean,NO2_Clean,SO2_Clean,CO_Clean,Data_Valid_Flag
Datetime,,,,,,,,,,
2017-11-24 17:00:00,AP001,NaN,NaN,60.50,98.00,126.40,30.80,11.85,0.10,1
2017-11-24 18:00:00,AP001,NaN,NaN,65.50,111.25,117.12,24.20,13.17,0.10,1
2017-11-24 19:00:00,AP001,NaN,NaN,80.00,132.00,98.98,25.18,12.08,0.10,1
2017-11-24 20:00:00,AP001,NaN,NaN,81.50,133.25,112.20,16.25,10.47,0.10,1
2017-11-24 21:00:00,AP001,NaN,NaN,75.25,116.00,106.35,17.48,9.12,0.10,1
...,...,...,...,...,...,...,...,...,...,...
2020-06-30 20:00:00,WB013,59.0,Satisfactory,15.55,47.80,17.24,35.08,9.40,0.80,1
2020-06-30 21:00:00,WB013,59.0,Satisfactory,15.23,42.30,17.46,26.78,4.91,0.56,1
2020-06-30 22:00:00,WB013,59.0,Satisfactory,11.40,40.95,17.24,19.53,3.81,0.61,1


In [12]:
#CO is in mg/m3; all others are in ug/m3.
AQI_BREAKPOINTS = {
    (0, 50): [  # Good
        (0, 30.0, 'PM2.5'), (0, 50.0, 'PM10'), (0, 40.0, 'NO2'), (0, 60.0, 'O3'),
        (0, 1.0, 'CO'), (0, 40.0, 'SO2')
    ],
    (51, 100): [ # Satisfactory
        (31.0, 60.0, 'PM2.5'), (51.0, 100.0, 'PM10'), (41.0, 80.0, 'NO2'), (61.0, 100.0, 'O3'),
        (1.1, 2.0, 'CO'), (41.0, 80.0, 'SO2')
    ],
    (101, 200): [ # Moderate
        (61.0, 90.0, 'PM2.5'), (101.0, 250.0, 'PM10'), (81.0, 180.0, 'NO2'), (101.0, 168.0, 'O3'),
        (2.1, 10.0, 'CO'), (81.0, 380.0, 'SO2')
    ],
    (201, 300): [ # Poor
        (91.0, 120.0, 'PM2.5'), (251.0, 350.0, 'PM10'), (181.0, 280.0, 'NO2'), (169.0, 208.0, 'O3'),
        (10.1, 17.0, 'CO'), (381.0, 800.0, 'SO2')
    ],
    (301, 400): [ # Very Poor
        (121.0, 250.0, 'PM2.5'), (351.0, 430.0, 'PM10'), (281.0, 400.0, 'NO2'), (209.0, 748.0, 'O3'),
        (17.1, 34.0, 'CO'), (801.0, 1600.0, 'SO2')
    ],
    (401, 500): [ # Severe
        (251.0, 9999.0, 'PM2.5'), (431.0, 9999.0, 'PM10'), (401.0, 9999.0, 'NO2'), (749.0, 9999.0, 'O3'),
        (34.1, 9999.0, 'CO'), (1601.0, 9999.0, 'SO2')
    ]
}

In [13]:
def get_pollutant_index(pollutant_name):
    pollutant_order = ['PM2.5', 'PM10',	'NO2',	'CO',	'SO2',	'O3']
    try:
        return pollutant_order.index(pollutant_name)
    except ValueError:
        return -1

def calculate_sub_index(pollutant_conc, pollutant_name, breakpoints):
    if pd.isna(pollutant_conc) or pollutant_conc < 0:
        return np.nan

    map_index = get_pollutant_index(pollutant_name)
    if map_index == -1:
        return np.nan

    for (I_L, I_H), pollutant_ranges in breakpoints.items():
        try:
            B_L, B_H, _ = pollutant_ranges[map_index]
        except IndexError:
            continue

        if pollutant_conc >= B_L and pollutant_conc <= B_H:
            # Ip = [(I_H - I_L) / (B_H - B_L)] * (C_p - B_L) + I_L
            if B_H == B_L:
                 return I_H

            Ip = ((I_H - I_L) / (B_H - B_L)) * (pollutant_conc - B_L) + I_L
            return Ip

        if I_H == 500 and pollutant_conc > B_H:
            return 500.0

    return np.nan

def determine_aqi_bucket(aqi_value):
    if pd.isna(aqi_value):
        return np.nan
    elif aqi_value <= 50:
        return 'Good'
    elif aqi_value <= 100:
        return 'Satisfactory'
    elif aqi_value <= 200:
        return 'Moderate'
    elif aqi_value <= 300:
        return 'Poor'
    elif aqi_value <= 400:
        return 'Very Poor'
    else:
        return 'Severe'

def calculate_overall_aqi_and_bucket(row, breakpoints):
    sub_indices = []

    pollutants_to_check = ['PM2.5', 'PM10',	'NO2',	'CO',	'SO2',	'O3']

    for p_name in pollutants_to_check:
        p_conc = row.get(p_name)
        Ip = calculate_sub_index(p_conc, p_name, breakpoints)
        if not pd.isna(Ip):
            sub_indices.append(Ip)

    if not sub_indices:
        return np.nan, np.nan

    final_aqi = max(sub_indices)
    aqi_bucket = determine_aqi_bucket(final_aqi)

    return final_aqi, aqi_bucket

In [14]:
df_hourly_cleaned.rename(columns=lambda x: x.replace('_Clean', ''), inplace = True)

In [15]:
df_hourly_cleaned = df_hourly_cleaned.drop(columns=['AQI', 'AQI_Bucket'], errors='ignore')

df_hourly_cleaned[['AQI', 'AQI_Bucket']] = df_hourly_cleaned.apply(
    lambda row: calculate_overall_aqi_and_bucket(row, AQI_BREAKPOINTS),
    axis=1,
    result_type='expand'
)

In [16]:
df_hourly_cleaned

,StationId,PM2.5,PM10,O3,NO2,SO2,CO,Data_Valid_Flag,AQI,AQI_Bucket
Datetime,,,,,,,,,,
2017-11-24 17:00:00,AP001,60.50,98.00,126.40,30.80,11.85,0.10,1,226.108696,Poor
2017-11-24 18:00:00,AP001,65.50,111.25,117.12,24.20,13.17,0.10,1,245.047826,Poor
2017-11-24 19:00:00,AP001,80.00,132.00,98.98,25.18,12.08,0.10,1,229.408696,Poor
2017-11-24 20:00:00,AP001,81.50,133.25,112.20,16.25,10.47,0.10,1,206.308696,Poor
2017-11-24 21:00:00,AP001,75.25,116.00,106.35,17.48,9.12,0.10,1,188.972152,Moderate
...,...,...,...,...,...,...,...,...,...,...
2020-06-30 20:00:00,WB013,15.55,47.80,17.24,35.08,9.40,0.80,1,192.481013,Moderate
2020-06-30 21:00:00,WB013,15.23,42.30,17.46,26.78,4.91,0.56,1,136.213924,Moderate
2020-06-30 22:00:00,WB013,11.40,40.95,17.24,19.53,3.81,0.61,1,122.429114,Moderate


In [17]:
df_hourly_cleaned.to_csv('/content/drive/MyDrive/impact_metrics/station_hour_imp.csv', index = True)